# 03 - Paraphrase COTs

Generate all paraphrase and transform variants of the COTs from notebook 02.

- Light paraphrase (via PARAPHRASER_MODEL)
- Heavy paraphrase (via PARAPHRASER_MODEL)
- Shuffled steps (Python, no LLM)
- Corrupted numbers (Python, no LLM)

**Fully resumable** - caches one file per (condition, problem).

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json

# Load all COTs from notebook 02
cots = []
for p in sorted(COT_CACHE.glob("*.json")):
    cots.append(json.loads(p.read_text()))

print(f"Loaded {len(cots)} COTs")
if len(cots) == 0:
    raise RuntimeError("No COTs found. Run 02_generate_cots.ipynb first.")

## Purge contaminated cache files

Delete paraphrase files that contain `<think>` tags (generated before the `enable_thinking=False` fix).
Also delete dependent prefill results so they get regenerated with clean paraphrases.

In [ ]:
import json
from pathlib import Path

# --- Purge contaminated paraphrase files (containing <think> tags) ---
paraphrase_conditions = ["paraphrase_light", "paraphrase_heavy"]
purged_paraphrases = 0

for cond in paraphrase_conditions:
    for p in PARAPHRASE_CACHE.glob(f"{cond}_*.json"):
        data = json.loads(p.read_text())
        if "<think>" in data.get("paraphrased_cot", ""):
            p.unlink()
            purged_paraphrases += 1

print(f"Purged {purged_paraphrases} contaminated paraphrase files")

# --- Delete dependent prefill results that used contaminated paraphrases ---
dependent_conditions = [
    "paraphrase_self", "paraphrase_cross",
    "heavy_paraphrase_self", "heavy_paraphrase_cross",
]
purged_prefills = 0

for cond in dependent_conditions:
    for p in PREFILL_CACHE.glob(f"{cond}_*.json"):
        p.unlink()
        purged_prefills += 1

# Also purge self_prefill (needs re-running with think-tag wrapper fix)
for p in PREFILL_CACHE.glob("self_prefill_*.json"):
    p.unlink()
    purged_prefills += 1

# Also purge no_cot (needs re-running with enable_thinking=False)
for p in PREFILL_CACHE.glob("no_cot_*.json"):
    p.unlink()
    purged_prefills += 1

print(f"Purged {purged_prefills} dependent prefill result files")
print("Cache is clean. Re-run notebooks 02 (no_cot cell), 03 (paraphrase cells), and 04.")

## Phase 1: Deterministic transforms (no GPU needed)

Shuffled steps and corrupted numbers are pure Python transforms.

In [ ]:
from lib.transforms import generate_transforms

generate_transforms(cots)

n_shuffle = len(list(PARAPHRASE_CACHE.glob("shuffle_*.json")))
n_corrupt = len(list(PARAPHRASE_CACHE.glob("corrupt_numbers_*.json")))
print(f"Shuffled steps: {n_shuffle} cached")
print(f"Corrupted numbers: {n_corrupt} cached")

## Phase 2: LLM paraphrases

Load PARAPHRASER_MODEL (Qwen3-8B) and generate light + heavy paraphrases.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

print(f"Loading {PARAPHRASER_MODEL}...")
llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
print("Paraphraser model loaded.")

In [ ]:
from lib.paraphrase import paraphrase_cots

# Light paraphrase
paraphrase_cots(llm, tokenizer, cots, condition="paraphrase_light", heavy=False)

In [ ]:
# Heavy paraphrase
paraphrase_cots(llm, tokenizer, cots, condition="paraphrase_heavy", heavy=True)

## Validation

In [ ]:
# Count cached files per condition
for condition in ["paraphrase_light", "paraphrase_heavy", "shuffle", "corrupt_numbers"]:
    n = len(list(PARAPHRASE_CACHE.glob(f"{condition}_*.json")))
    print(f"{condition}: {n} cached")

# Show a sample light paraphrase
sample = json.loads(next(PARAPHRASE_CACHE.glob("paraphrase_light_*.json")).read_text())
print(f"\n--- Original COT (first 300 chars) ---")
print(sample["original_cot"][:300])
print(f"\n--- Paraphrased COT (first 300 chars) ---")
print(sample["paraphrased_cot"][:300])

In [ ]:
# Clean up
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("Paraphraser model unloaded.")